In [ ]:
from google.colab import drive

drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
import os

print(os.listdir('/content/drive/MyDrive')[:30])

['Colab Notebooks', 'Resume .pdf', 'bhuasw.m3u8', 'Classroom', 'Untitled presentation (1).gslides', 'ALVEOLAR BONE GRAFTING.gslides', 'Untitled presentation.gslides', 'MANAGEMENT OF N0 NECK.gslides', 'PI_KIT 2025.pdf', 'Balance amount Term-1 fees.pdf', 'image.jpg', 'photo_2025-07-05_23-11-50.jpg', 'Excel worksheet.gsheet', 'Copy of [MAKE A COPY] Worksheet: SEO Essentials with Semrush - 101.gsheet', '1782755183656494.mov', 'Content Enrichment Workshop: PGT Maths (KVS) (2).gslides', 'Content Enrichment Workshop: PGT Maths (KVS) (1).gslides', 'Content Enrichment Workshop: PGT Maths (KVS).gslides', 'raw', 'processed']


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/raw")

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)

Project: /content/drive/MyDrive/raw
Raw: /content/drive/MyDrive/raw/data/raw
Processed: /content/drive/MyDrive/raw/data/processed


In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file == 'Electronics.jsonl.gz':
            print("FOUND:")
            print(os.path.join(root, file))

FOUND:
/content/drive/MyDrive/raw/Electronics.jsonl.gz


In [ ]:
from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/raw")
PROCESSED_DIR = Path("/content/drive/MyDrive/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/drive/MyDrive/raw
Processed folder: /content/drive/MyDrive/processed


In [ ]:
print(list(RAW_DIR.iterdir())[:20])

In [ ]:
!pip -q install duckdb

In [ ]:
import duckdb

print("DuckDB ready")

In [ ]:
CATEGORY = "Electronics"

REVIEW_FILE = RAW_DIR / "Electronics.jsonl.gz"

print("File:", REVIEW_FILE)
print("Exists:", REVIEW_FILE.exists())

File: /content/drive/MyDrive/raw/Electronics.jsonl.gz
Exists: True


In [ ]:
import os

TEMP_DIR = "/content/amazon_recommender_temp"
os.makedirs(TEMP_DIR, exist_ok=True)

print(TEMP_DIR)

In [ ]:
 DB_FILE = f"{TEMP_DIR}/Electronics.duckdb"

con = duckdb.connect(DB_FILE)

con.execute("SET memory_limit='8GB'")
con.execute("SET threads=4")
con.execute("SET preserve_insertion_order=false")
con.execute(f"SET temp_directory='{TEMP_DIR}'")

print("DuckDB connection successful")

DuckDB connection successful


In [ ]:
 print("Starting Electronics processing...")
print("This may take some time because the file is very large.")

con.execute(f"""
CREATE OR REPLACE TABLE interactions AS

SELECT
    user_id,
    parent_asin AS product_id,
    CAST(rating AS DOUBLE) AS rating,
    CAST(timestamp AS BIGINT) AS timestamp

FROM read_json(
    '{REVIEW_FILE}',
    format='newline_delimited',
    columns={{
        'user_id': 'VARCHAR',
        'parent_asin': 'VARCHAR',
        'rating': 'DOUBLE',
        'timestamp': 'BIGINT'
    }}
)

WHERE user_id IS NOT NULL
  AND parent_asin IS NOT NULL
  AND rating >= 1
  AND rating <= 5
""")

print("✅ Raw Electronics data processed")

Starting Electronics processing...
This may take some time because the file is very large.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Raw Electronics data processed


In [ ]:
print("Finding users with at least 5 interactions...")

con.execute("""
CREATE OR REPLACE TABLE active_users AS

SELECT
    user_id,
    COUNT(*) AS interaction_count

FROM interactions

GROUP BY user_id

HAVING COUNT(*) >= 5
""")

active_users = con.execute("""
SELECT COUNT(*)
FROM active_users
""").fetchone()[0]

print("Users with >= 5 interactions:", active_users)

In [ ]:
print("Finding products with at least 5 interactions...")

con.execute("""
CREATE OR REPLACE TABLE active_products AS

SELECT
    product_id,
    COUNT(*) AS interaction_count

FROM interactions

GROUP BY product_id

HAVING COUNT(*) >= 5
""")

active_products = con.execute("""
SELECT COUNT(*)
FROM active_products
""").fetchone()[0]

print("Products with >= 5 interactions:", active_products)

Finding products with at least 5 interactions...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Products with >= 5 interactions: 625507


In [ ]:
print("Creating filtered interaction dataset...")

con.execute("""
CREATE OR REPLACE TABLE filtered_interactions AS

SELECT
    i.user_id,
    i.product_id,
    i.rating,
    i.timestamp

FROM interactions i

INNER JOIN active_users u
    ON i.user_id = u.user_id

INNER JOIN active_products p
    ON i.product_id = p.product_id
""")

filtered_count = con.execute("""
SELECT COUNT(*)
FROM filtered_interactions
""").fetchone()[0]

print("Filtered interactions:", filtered_count)

Creating filtered interaction dataset...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filtered interactions: 17273814


In [ ]:
print("Original interactions:",
      con.execute("SELECT COUNT(*) FROM interactions").fetchone()[0])

print("Active users:", active_users)

print("Active products:", active_products)

print("Final eligible interactions:", filtered_count)

Original interactions: 43886942
Active users: 1881540
Active products: 625507
Final eligible interactions: 17273814


In [ ]:
TARGET = 400_000

output_file = PROCESSED_DIR / "Electronics_interactions.csv"

print(f"Selecting {TARGET:,} interactions...")
print(f"Available eligible interactions: {filtered_count:,}")

con.execute(f"""
COPY (

    SELECT
        user_id,
        product_id,
        rating,
        timestamp

    FROM filtered_interactions

    ORDER BY hash(
        user_id || '|' ||
        product_id || '|' ||
        CAST(timestamp AS VARCHAR)
    )

    LIMIT {TARGET}

) TO '{output_file}'
WITH (
    HEADER,
    DELIMITER ','
)
""")

print("\n✅ Electronics sample created!")
print("Saved to:")
print(output_file)

Selecting 400,000 interactions...
Available eligible interactions: 17,273,814


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✅ Electronics sample created!
Saved to:
/content/drive/MyDrive/processed/Electronics_interactions.csv


In [ ]:
import pandas as pd

df = pd.read_csv(output_file)

print("Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nNumber of users:", df["user_id"].nunique())
print("Number of products:", df["product_id"].nunique())

print("\nRating distribution:")
print(df["rating"].value_counts().sort_index())

print("\nMissing values:")
print(df.isnull().sum())

Shape: (400000, 4)

First 5 rows:


,user_id,product_id,rating,timestamp
0,AETI7KRRIXZ5RCAMJBJDFZTVTF2A,B0B2JYYR6C,5.0,1592320299841
1,AFQXB6XDWDBJ3V7MQYMSNDVKAQWA,B07GCKFL4G,5.0,1569809511701
2,AHJ4KBDI6M6X4OIIQF7BQO3WUI5A,B009J90TR2,3.0,1448303081000
3,AFLNWZLN5NFHF5WJUVSW3LHVDGZA,B000AQF4JG,2.0,1263416280000
4,AGE7MEIPCLPFTGIMTPVXGKN5L2TA,B00ZOU93Q8,1.0,1490406882000



Number of users: 338220
Number of products: 149351

Rating distribution:
rating
1.0     34287
2.0     18816
3.0     27578
4.0     56033
5.0    263286
Name: count, dtype: int64

Missing values:
user_id       0
product_id    0
rating        0
timestamp     0
dtype: int64


In [ ]:
import duckdb
import os
from pathlib import Path

# ============================================================
# 1. PATHS
# ============================================================

RAW_DIR = Path("/content/drive/MyDrive/raw")
PROCESSED_DIR = Path("/content/drive/MyDrive/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Temporary processing happens on Colab, NOT Google Drive
TEMP_DIR = Path("/content/amazon_recommender_temp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. CATEGORIES AND TARGET SIZES
# ============================================================

CATEGORY_TARGETS = {
    "Automotive": 250_000,
    "Beauty_and_Personal_Care": 300_000,
    "Books": 300_000,
    "Clothing_Shoes_and_Jewelry": 400_000,
    "Electronics": 400_000,
    "Grocery_and_Gourmet_Food": 200_000,
    "Home_and_Kitchen": 400_000,
    "Office_Products": 200_000,
    "Sports_and_Outdoors": 300_000,
    "Toys_and_Games": 250_000
}


# ============================================================
# 3. CHECK FILES
# ============================================================

print("=" * 70)
print("CHECKING RAW FILES")
print("=" * 70)

missing_files = []

for category in CATEGORY_TARGETS:

    file_path = RAW_DIR / f"{category}.jsonl.gz"

    if file_path.exists():
        print(f"✓ {category}")
    else:
        print(f"✗ MISSING: {category}")
        missing_files.append(category)

if missing_files:
    raise FileNotFoundError(
        f"\nMissing files: {missing_files}\n"
        f"Check the filenames inside: {RAW_DIR}"
    )

print("\nAll 10 category files found.")


# ============================================================
# 4. CONNECT TO DUCKDB
# ============================================================

DB_FILE = TEMP_DIR / "amazon_recommender.duckdb"

con = duckdb.connect(str(DB_FILE))

con.execute("SET memory_limit='8GB'")
con.execute("SET threads=4")
con.execute("SET preserve_insertion_order=false")
con.execute(f"SET temp_directory='{TEMP_DIR}'")


# ============================================================
# 5. PROCESS EACH CATEGORY
# ============================================================

summary = []

for category, target in CATEGORY_TARGETS.items():

    print("\n")
    print("=" * 70)
    print(f"PROCESSING: {category}")
    print(f"TARGET: {target:,} interactions")
    print("=" * 70)

    review_file = RAW_DIR / f"{category}.jsonl.gz"

    # --------------------------------------------------------
    # Remove old temporary tables
    # --------------------------------------------------------

    con.execute("DROP TABLE IF EXISTS interactions")
    con.execute("DROP TABLE IF EXISTS active_users")
    con.execute("DROP TABLE IF EXISTS active_products")
    con.execute("DROP TABLE IF EXISTS filtered_interactions")


    # --------------------------------------------------------
    # Step 1: Read raw reviews
    # --------------------------------------------------------

    print("\n[1/5] Reading raw reviews...")

    con.execute(f"""
        CREATE TABLE interactions AS

        SELECT
            user_id,
            parent_asin AS product_id,
            CAST(rating AS DOUBLE) AS rating,
            CAST(timestamp AS BIGINT) AS timestamp

        FROM read_json(
            '{review_file}',
            format='newline_delimited',
            columns={{
                'user_id': 'VARCHAR',
                'parent_asin': 'VARCHAR',
                'rating': 'DOUBLE',
                'timestamp': 'BIGINT'
            }}
        )

        WHERE user_id IS NOT NULL
          AND parent_asin IS NOT NULL
          AND rating >= 1
          AND rating <= 5
    """)

    original_count = con.execute("""
        SELECT COUNT(*)
        FROM interactions
    """).fetchone()[0]

    print(f"    Valid interactions: {original_count:,}")


    # --------------------------------------------------------
    # Step 2: Active users
    # --------------------------------------------------------

    print("\n[2/5] Finding active users (>=5 interactions)...")

    con.execute("""
        CREATE TABLE active_users AS

        SELECT
            user_id,
            COUNT(*) AS interaction_count

        FROM interactions

        GROUP BY user_id

        HAVING COUNT(*) >= 5
    """)

    active_user_count = con.execute("""
        SELECT COUNT(*)
        FROM active_users
    """).fetchone()[0]

    print(f"    Active users: {active_user_count:,}")


    # --------------------------------------------------------
    # Step 3: Active products
    # --------------------------------------------------------

    print("\n[3/5] Finding active products (>=5 interactions)...")

    con.execute("""
        CREATE TABLE active_products AS

        SELECT
            product_id,
            COUNT(*) AS interaction_count

        FROM interactions

        GROUP BY product_id

        HAVING COUNT(*) >= 5
    """)

    active_product_count = con.execute("""
        SELECT COUNT(*)
        FROM active_products
    """).fetchone()[0]

    print(f"    Active products: {active_product_count:,}")


    # --------------------------------------------------------
    # Step 4: Keep interactions from active users/products
    # --------------------------------------------------------

    print("\n[4/5] Creating eligible interactions...")

    con.execute("""
        CREATE TABLE filtered_interactions AS

        SELECT
            i.user_id,
            i.product_id,
            i.rating,
            i.timestamp

        FROM interactions i

        INNER JOIN active_users u
            ON i.user_id = u.user_id

        INNER JOIN active_products p
            ON i.product_id = p.product_id
    """)

    filtered_count = con.execute("""
        SELECT COUNT(*)
        FROM filtered_interactions
    """).fetchone()[0]

    print(f"    Eligible interactions: {filtered_count:,}")


    # --------------------------------------------------------
    # Step 5: Sample target number
    # --------------------------------------------------------

    actual_target = min(target, filtered_count)

    print(f"\n[5/5] Selecting {actual_target:,} interactions...")

    output_file = PROCESSED_DIR / f"{category}_interactions.csv"

    con.execute(f"""
        COPY (

            SELECT
                user_id,
                product_id,
                rating,
                timestamp

            FROM filtered_interactions

            ORDER BY hash(
                user_id || '|' ||
                product_id || '|' ||
                CAST(timestamp AS VARCHAR)
            )

            LIMIT {actual_target}

        ) TO '{output_file}'

        WITH (
            HEADER,
            DELIMITER ','
        )
    """)

    print(f"    ✓ Saved: {output_file}")

    summary.append({
        "category": category,
        "original_interactions": original_count,
        "active_users": active_user_count,
        "active_products": active_product_count,
        "eligible_interactions": filtered_count,
        "sampled_interactions": actual_target
    })


# ============================================================
# 6. CLOSE DUCKDB
# ============================================================

con.close()

print("\n")
print("=" * 70)
print("ALL 10 CATEGORIES PROCESSED")
print("=" * 70)

CHECKING RAW FILES
✓ Automotive
✓ Beauty_and_Personal_Care
✓ Books
✓ Clothing_Shoes_and_Jewelry
✓ Electronics
✓ Grocery_and_Gourmet_Food
✓ Home_and_Kitchen
✓ Office_Products
✓ Sports_and_Outdoors
✓ Toys_and_Games

All 10 category files found.


PROCESSING: Automotive
TARGET: 250,000 interactions

[1/5] Reading raw reviews...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
import pandas as pd
from pathlib import Path

print("Combining category datasets...")

all_files = []

for category in CATEGORY_TARGETS:

    file_path = PROCESSED_DIR / f"{category}_interactions.csv"

    if file_path.exists():
        all_files.append(file_path)
        print(f"✓ {category}")
    else:
        print(f"✗ Missing: {category}")


# Read and combine
dfs = []

for file_path in all_files:
    df_temp = pd.read_csv(file_path)
    dfs.append(df_temp)

final_df = pd.concat(dfs, ignore_index=True)

# Add category information
category_lookup = {}

for category in CATEGORY_TARGETS:
    category_lookup[category] = category

# Category can be recovered from the individual files,
# so rebuild final dataset with category column.

dfs = []

for category in CATEGORY_TARGETS:

    file_path = PROCESSED_DIR / f"{category}_interactions.csv"

    df_temp = pd.read_csv(file_path)

    df_temp["category"] = category

    dfs.append(df_temp)

final_df = pd.concat(dfs, ignore_index=True)

# Save final dataset
FINAL_FILE = PROCESSED_DIR / "interactions.csv"

final_df.to_csv(FINAL_FILE, index=False)

print("\n" + "=" * 70)
print("FINAL DATASET CREATED")
print("=" * 70)

print("File:", FINAL_FILE)
print("Rows:", len(final_df))
print("Users:", final_df["user_id"].nunique())
print("Products:", final_df["product_id"].nunique())

print("\nCategory distribution:")
print(final_df["category"].value_counts())

print("\nRating distribution:")
print(final_df["rating"].value_counts().sort_index())

Combining category datasets...
✓ Automotive
✓ Beauty_and_Personal_Care
✓ Books
✓ Clothing_Shoes_and_Jewelry
✓ Electronics
✓ Grocery_and_Gourmet_Food
✓ Home_and_Kitchen
✓ Office_Products
✓ Sports_and_Outdoors
✓ Toys_and_Games

FINAL DATASET CREATED
File: /content/drive/MyDrive/processed/interactions.csv
Rows: 3000000
Users: 1908442
Products: 1372495

Category distribution:
category
Electronics                   400000
Clothing_Shoes_and_Jewelry    400000
Home_and_Kitchen              400000
Beauty_and_Personal_Care      300000
Sports_and_Outdoors           300000
Books                         300000
Automotive                    250000
Toys_and_Games                250000
Grocery_and_Gourmet_Food      200000
Office_Products               200000
Name: count, dtype: int64

Rating distribution:
rating
1.0     196586
2.0     126724
3.0     216565
4.0     408244
5.0    2051881
Name: count, dtype: int64


In [ ]:
import pandas as pd

FINAL_FILE = "/content/drive/MyDrive/processed/interactions.csv"

df = pd.read_csv(FINAL_FILE)

print("=" * 60)
print("FINAL INTERACTION DATASET")
print("=" * 60)

print("\nShape:")
print(df.shape)

print("\nUnique users:")
print(df["user_id"].nunique())

print("\nUnique products:")
print(df["product_id"].nunique())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nRating distribution:")
print(df["rating"].value_counts().sort_index())

print("\nCategory distribution:")
print(df["category"].value_counts())

print("\nInteractions per user:")
print(df.groupby("user_id").size().describe())

print("\nInteractions per product:")
print(df.groupby("product_id").size().describe())

FINAL INTERACTION DATASET

Shape:
(3000000, 5)

Unique users:
1908442

Unique products:
1372495

Missing values:
user_id       0
product_id    0
rating        0
timestamp     0
category      0
dtype: int64

Duplicate rows:
43385

Rating distribution:
rating
1.0     196586
2.0     126724
3.0     216565
4.0     408244
5.0    2051881
Name: count, dtype: int64

Category distribution:
category
Electronics                   400000
Clothing_Shoes_and_Jewelry    400000
Home_and_Kitchen              400000
Beauty_and_Personal_Care      300000
Sports_and_Outdoors           300000
Books                         300000
Automotive                    250000
Toys_and_Games                250000
Grocery_and_Gourmet_Food      200000
Office_Products               200000
Name: count, dtype: int64

Interactions per user:
count    1.908442e+06
mean     1.571963e+00
std      1.664125e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      3.120000e+02
dtype: float

In [ ]:
from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/raw")

for file in RAW_DIR.iterdir():
    if file.name.startswith("meta_"):
        print(file.name)

meta_Office_Products.jsonl.gz
meta_Grocery_and_Gourmet_Food.jsonl.gz
meta_Automotive.jsonl.gz
meta_Toys_and_Games.jsonl.gz
meta_Sports_and_Outdoors.jsonl.gz
meta_Beauty_and_Personal_Care.jsonl.gz
meta_Home_and_Kitchen.jsonl.gz
meta_Electronics.jsonl.gz
meta_Books.jsonl.gz
meta_Clothing_Shoes_and_Jewelry.jsonl.gz


In [ ]:
import gzip
import json
from pathlib import Path

META_FILE = Path(
    "/content/drive/MyDrive/raw/meta_Electronics.jsonl.gz"
)

with gzip.open(META_FILE, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        product = json.loads(line)

        print("PRODUCT RECORD:")
        print(product)

        break

PRODUCT RECORD:
{'main_category': 'All Electronics', 'title': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET', 'average_rating': 3.5, 'rating_number': 6, 'features': [], 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'], 'price': None, 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL

In [ ]:
{
    'main_category': ...,
    'title': ...,
    'average_rating': ...,
    'rating_number': ...,
    'features': ...,
    'description': ...,
    'price': ...,
    'images': ...,
    'videos': ...,
    'store': ...,
    'categories': ...,
    'details': ...,
    'parent_asin': ...
}

{'main_category': Ellipsis,
 'title': Ellipsis,
 'average_rating': Ellipsis,
 'rating_number': Ellipsis,
 'features': Ellipsis,
 'description': Ellipsis,
 'price': Ellipsis,
 'images': Ellipsis,
 'videos': Ellipsis,
 'store': Ellipsis,
 'categories': Ellipsis,
 'details': Ellipsis,
 'parent_asin': Ellipsis}

In [ ]:
import gzip
import json
import pandas as pd
from pathlib import Path

# ==============================
# PATHS
# ==============================

RAW_DIR = Path("/content/drive/MyDrive/raw")
PROCESSED_DIR = Path("/content/drive/MyDrive/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


# ==============================
# CATEGORIES
# ==============================

CATEGORIES = [
    "Electronics",
    "Home_and_Kitchen",
    "Clothing_Shoes_and_Jewelry",
    "Beauty_and_Personal_Care",
    "Sports_and_Outdoors",
    "Toys_and_Games",
    "Automotive",
    "Books",
    "Grocery_and_Gourmet_Food",
    "Office_Products"
]


# ==============================
# FUNCTION TO PROCESS ONE FILE
# ==============================

def process_metadata(category):

    file_path = RAW_DIR / f"meta_{category}.jsonl.gz"

    print(f"\nProcessing: {category}")
    print(f"File: {file_path}")

    products = []

    with gzip.open(file_path, "rt", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            try:
                item = json.loads(line)

                product_id = item.get("parent_asin")

                if not product_id:
                    continue

                # Convert list fields into text
                features = item.get("features", [])
                description = item.get("description", [])

                if isinstance(features, list):
                    features = " ".join(
                        str(x) for x in features if x
                    )

                if isinstance(description, list):
                    description = " ".join(
                        str(x) for x in description if x
                    )

                products.append({
                    "product_id": product_id,
                    "title": item.get("title"),
                    "category": category,
                    "average_rating": item.get("average_rating"),
                    "rating_number": item.get("rating_number"),
                    "features": features,
                    "description": description,
                    "price": item.get("price"),
                    "store": item.get("store")
                })

            except Exception:
                continue

            # Progress update
            if line_number % 500000 == 0:
                print(f"Processed {line_number:,} records...")

    df = pd.DataFrame(products)

    print(f"Products extracted: {len(df):,}")

    return df


# ==============================
# PROCESS ALL 10 CATEGORIES
# ==============================

all_products = []

for category in CATEGORIES:

    try:
        category_df = process_metadata(category)
        all_products.append(category_df)

    except FileNotFoundError:
        print(f"\n⚠️ FILE NOT FOUND: meta_{category}.jsonl.gz")


# ==============================
# COMBINE EVERYTHING
# ==============================

products_df = pd.concat(
    all_products,
    ignore_index=True
)


# ==============================
# REMOVE DUPLICATE PRODUCTS
# ==============================

products_df = products_df.drop_duplicates(
    subset=["product_id"],
    keep="first"
)


# ==============================
# SAVE
# ==============================

output_path = PROCESSED_DIR / "products.csv"

products_df.to_csv(
    output_path,
    index=False
)


print("\n" + "=" * 60)
print("PRODUCT DATASET CREATED")
print("=" * 60)

print(f"\nShape: {products_df.shape}")
print(f"Unique products: {products_df['product_id'].nunique():,}")

print("\nMissing values:")
print(products_df.isnull().sum())

print(f"\nSaved to:\n{output_path}")


Processing: Electronics
File: /content/drive/MyDrive/raw/meta_Electronics.jsonl.gz
Processed 500,000 records...
Processed 1,000,000 records...
Processed 1,500,000 records...
Products extracted: 1,610,012

Processing: Home_and_Kitchen
File: /content/drive/MyDrive/raw/meta_Home_and_Kitchen.jsonl.gz
Processed 500,000 records...
Processed 1,000,000 records...
Processed 1,500,000 records...
Processed 2,000,000 records...
Processed 2,500,000 records...
Processed 3,000,000 records...
Processed 3,500,000 records...
Products extracted: 3,735,584

Processing: Clothing_Shoes_and_Jewelry
File: /content/drive/MyDrive/raw/meta_Clothing_Shoes_and_Jewelry.jsonl.gz
Processed 500,000 records...
